<a href="https://colab.research.google.com/github/CMILINAZZO/medical-llm-self-bias-audit/blob/main/notebooks/03_deepeval_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Setup & Installations
# Install DeepEval and the official SDKs for our judge models.
!pip install -q deepeval openai anthropic google-genai pandas tqdm

In [ ]:
# Cell 2: Secure Credentials
import os
from google.colab import userdata

# DeepEval automatically looks for these specific environment variables when initializing default models.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

print("✓ API Keys successfully loaded into environment variables for DeepEval.")

In [ ]:
# Cell 3: Repository Sync & Path Setup
import sys
import shutil
from pathlib import Path
import pandas as pd

# 1. Detect runtime environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Environment Detected: Google Colab")

    # Configuration
    GITHUB_USERNAME = "CMILINAZZO"
    REPO_NAME = "medical-llm-self-bias-audit"
    REPO_ROOT = Path(f"/content/{REPO_NAME}")

    # 2. Check for fake or corrupted non-git directories
    if REPO_ROOT.exists() and not (REPO_ROOT / ".git").exists():
        print("Found an empty or plain folder block where a Git repo should be. Clearing it...")
        shutil.rmtree(REPO_ROOT)

    # 3. Clean clone or pull sequence
    if not REPO_ROOT.exists():
        print(f"Cloning fresh copy of the public repository from GitHub...")
        !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
    else:
        print(f"Active Git repository found. Pulling latest upstream updates...")
        current_dir = os.getcwd()
        os.chdir(REPO_ROOT)
        !git pull
        os.chdir(current_dir)

    # 4. Synchronize Python's working directory
    os.chdir(REPO_ROOT / "notebooks")
    print(f"\n✓ Active Working Directory synchronized to: {os.getcwd()}")

# 5. Standardized portable paths
INPUT_PATH = "../outputs/generation_results.csv"
OUTPUT_PATH = "../outputs/evaluated_results_api.csv"

# Load the generated student data
df = pd.read_csv(INPUT_PATH)
print(f"✓ Data loaded successfully. Ready to evaluate {df.shape[0]} rows.")